# 02 数据清洗与存储

本Notebook负责：
- 清洗股票数据（缺失值、重复值、离群值等）
- 宽表与长表转换
- 多表合并
- 存储到CSV和SQLite数据库

In [20]:
import pandas as pd
import numpy as np
import os
import sqlite3
from datetime import datetime

# 设置项目根目录
PROJECT_ROOT = os.path.dirname(os.path.abspath('.')) if os.path.basename(os.path.abspath('.')) == 'dshw-p02' else os.path.abspath('.')
os.chdir(PROJECT_ROOT)
print(f"当前工作目录: {os.getcwd()}")

# 股票列表
stocks = [
    {"code": "sh.600036", "name": "招商银行", "industry": "银行"},
    {"code": "sh.601328", "name": "交通银行", "industry": "银行"},
    {"code": "sz.002594", "name": "比亚迪", "industry": "汽车"},
    {"code": "sh.600104", "name": "上汽集团", "industry": "汽车"},
    {"code": "sz.000002", "name": "万科A", "industry": "房地产"},
    {"code": "sh.600048", "name": "保利发展", "industry": "房地产"},
    {"code": "sh.600519", "name": "贵州茅台", "industry": "白酒"},
    {"code": "sz.000858", "name": "五粮液", "industry": "白酒"},
    {"code": "sh.601012", "name": "隆基绿能", "industry": "能源"},
    {"code": "sh.600028", "name": "中国石化", "industry": "能源"}
]

当前工作目录: d:\2026研一下\dsfin-hw


## 3.1 单表清洗

In [21]:
def clean_single_stock(df, stock_code, stock_name):
    """清洗单只股票的数据"""
    print(f"\n========== 清洗 {stock_name} ({stock_code}) ==========")
    
    # 复制数据以避免修改原始数据
    df_clean = df.copy()
    
    # === 1. 缺失值检测 ===
    print("\n--- 缺失值检测 ---")
    missing_stats = []
    for col in df_clean.columns:
        missing_count = df_clean[col].isna().sum()
        missing_ratio = missing_count / len(df_clean) * 100
        missing_stats.append({
            '列名': col,
            '缺失数量': missing_count,
            '缺失比例(%)': round(missing_ratio, 2)
        })
    missing_df = pd.DataFrame(missing_stats)
    print(missing_df)
    
    # 缺失值可能原因：停牌、节假日、数据源问题
    # 这里选择向前填充（ffill），因为金融数据有时间序列连续性
    print("\n--- 缺失值处理：向前填充 ---")
    before_fill = df_clean.isna().sum().sum()
    df_clean = df_clean.ffill()
    # 剩余的缺失值用向后填充
    df_clean = df_clean.bfill()
    after_fill = df_clean.isna().sum().sum()
    print(f"填充前缺失值总数: {before_fill}, 填充后: {after_fill}")
    
    # === 2. 日期格式统一 ===
    print("\n--- 日期格式统一 ---")
    df_clean['date'] = pd.to_datetime(df_clean['date'])
    df_clean = df_clean.set_index('date').sort_index()
    print(f"日期范围: {df_clean.index.min()} 至 {df_clean.index.max()}")
    
    # === 3. 数据类型检查 ===
    print("\n--- 数据类型检查 ---")
    numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']
    for col in numeric_cols:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    print(df_clean.dtypes)
    
    # === 4. 重复值处理 ===
    print("\n--- 重复值处理 ---")
    before_dedup = len(df_clean)
    df_clean = df_clean[~df_clean.index.duplicated(keep='first')]
    after_dedup = len(df_clean)
    print(f"去重前行数: {before_dedup}, 去重后: {after_dedup}, 删除: {before_dedup - after_dedup}")
    
    # === 5. 离群值标注 ===
    print("\n--- 离群值标注 ---")
    # 计算日收益率
    df_clean['return'] = np.log(df_clean['close'] / df_clean['close'].shift(1))
    # 标注单日涨跌幅超过±20%的记录
    df_clean['is_extreme'] = abs(df_clean['return']) > 0.2
    extreme_count = df_clean['is_extreme'].sum()
    print(f"离群值（涨跌幅>20%）数量: {extreme_count}")
    if extreme_count > 0:
        print("离群值日期:")
        print(df_clean[df_clean['is_extreme']].index.tolist())
    
    return df_clean.reset_index()

# 清洗所有股票
cleaned_stocks = {}
for stock in stocks:
    code_simple = stock['code'].replace('.', '_')
    filepath = f"dshw-p02/data/stock/stock_{code_simple}.csv"
    
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        df_clean = clean_single_stock(df, stock['code'], stock['name'])
        cleaned_stocks[stock['code']] = df_clean
    else:
        print(f"文件不存在: {filepath}")
        # 生成模拟数据
        print(f"为 {stock['name']} 生成模拟数据...")
        dates = pd.date_range(start='2020-01-01', end='2026-04-07', freq='B')
        np.random.seed(hash(stock['code']) % 10000)
        base_price = np.random.uniform(10, 100)
        returns = np.random.normal(0, 0.02, len(dates))
        prices = base_price * np.cumprod(1 + returns)
        
        df_sim = pd.DataFrame({
            'date': dates,
            'open': prices * (1 + np.random.normal(0, 0.01, len(dates))),
            'high': prices * (1 + np.abs(np.random.normal(0, 0.02, len(dates)))),
            'low': prices * (1 - np.abs(np.random.normal(0, 0.02, len(dates)))),
            'close': prices,
            'volume': np.random.randint(1000000, 100000000, len(dates)),
            'amount': prices * np.random.randint(1000000, 100000000, len(dates))
        })
        df_clean = clean_single_stock(df_sim, stock['code'], stock['name'])
        cleaned_stocks[stock['code']] = df_clean


========== 清洗 招商银行 (sh.600036) ==========

--- 缺失值检测 ---
       列名  缺失数量  缺失比例(%)
0    date     0      0.0
1    open     0      0.0
2    high     0      0.0
3     low     0      0.0
4   close     0      0.0
5  volume     0      0.0
6  amount     0      0.0

--- 缺失值处理：向前填充 ---
填充前缺失值总数: 0, 填充后: 0

--- 日期格式统一 ---
日期范围: 2020-01-02 00:00:00 至 2026-04-07 00:00:00

--- 数据类型检查 ---
open      float64
high      float64
low       float64
close     float64
volume      int64
amount    float64
dtype: object

--- 重复值处理 ---
去重前行数: 1515, 去重后: 1515, 删除: 0

--- 离群值标注 ---
离群值（涨跌幅>20%）数量: 0

========== 清洗 交通银行 (sh.601328) ==========

--- 缺失值检测 ---
       列名  缺失数量  缺失比例(%)
0    date     0      0.0
1    open     0      0.0
2    high     0      0.0
3     low     0      0.0
4   close     0      0.0
5  volume     0      0.0
6  amount     0      0.0

--- 缺失值处理：向前填充 ---
填充前缺失值总数: 0, 填充后: 0

--- 日期格式统一 ---
日期范围: 2020-01-02 00:00:00 至 2026-04-07 00:00:00

--- 数据类型检查 ---
open      float64
high      float64
low     

## 3.2 宽表与长表转换

In [22]:
# 创建收盘价宽表
print("--- 创建收盘价宽表 ---")
close_wide = pd.DataFrame()

for stock in stocks:
    code = stock['code']
    if code in cleaned_stocks:
        df = cleaned_stocks[code]
        df = df.set_index('date')
        close_wide[stock['name']] = df['close']

print(f"宽表形状: {close_wide.shape}")
print("宽表前5行:")
print(close_wide.head())

# 转换为长表
print("\n--- 转换为长表 ---")
close_long = close_wide.reset_index().melt(
    id_vars=['date'], 
    var_name='stock_name', 
    value_name='close'
)

print(f"长表形状: {close_long.shape}")
print("长表前5行:")
print(close_long.head())

# 保存清洗后的股票数据
all_stocks_clean = []
for stock in stocks:
    code = stock['code']
    if code in cleaned_stocks:
        df = cleaned_stocks[code].copy()
        df['code'] = code
        df['name'] = stock['name']
        df['industry'] = stock['industry']
        all_stocks_clean.append(df)

stock_clean_df = pd.concat(all_stocks_clean, ignore_index=True)
stock_clean_df.to_csv("dshw-p02/data/clean/stock_clean.csv", index=False, encoding="utf-8-sig")
print(f"\n清洗后股票数据已保存至 data/clean/stock_clean.csv, 形状: {stock_clean_df.shape}")

print("\n【问题回答】")
print("宽表适合的操作：横向比较多只股票的价格，计算相关系数矩阵，可视化多只股票走势。")
print("长表适合的操作：按股票分组统计，绘制分面图，进行回归分析等需要按个体分组的操作。")

--- 创建收盘价宽表 ---
宽表形状: (1515, 10)
宽表前5行:
                 招商银行      交通银行        比亚迪       上汽集团        万科A       保利发展  \
date                                                                          
2020-01-02  29.432976  3.853710  15.582369  20.212888  26.594324  12.602703   
2020-01-03  29.826627  3.860494  15.540315  20.154348  26.177767  12.362430   
2020-01-06  29.705504  3.840140  15.617952  19.535501  25.736706  12.153160   
2020-01-07  29.637372  3.860494  15.543550  19.677668  25.940901  12.238418   
2020-01-08  29.077177  3.826571  15.294465  19.978729  25.875559  12.029148   

                  贵州茅台         五粮液       隆基绿能      中国石化  
date                                                     
2020-01-02  979.045560  111.862382  13.003149  3.524560  
2020-01-03  934.477327  110.566581  13.223707  3.592733  
2020-01-06  933.983472  109.423227  13.527588  3.654089  
2020-01-07  948.313926  109.567205  13.542292  3.579098  
2020-01-08  942.777554  109.160679  13.581502  3.599550  


## 3.3 多表合并

In [23]:
# 1. 加载指数数据
print("--- 加载指数数据 ---")
index_data_list = []

for index_code, index_name in [('sh.000300', '沪深300'), ('sh.000905', '中证500')]:
    code_simple = index_code.replace('.', '_')
    filepath = f"data/index/index_{code_simple}.csv"
    
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
    else:
        # 生成模拟指数数据
        dates = pd.date_range(start='2020-01-01', end='2026-04-07', freq='B')
        np.random.seed(hash(index_code) % 10000)
        base_price = 4000 if index_code == 'sh.000300' else 6000
        returns = np.random.normal(0, 0.015, len(dates))
        prices = base_price * np.cumprod(1 + returns)
        
        df = pd.DataFrame({
            'date': dates,
            'open': prices * (1 + np.random.normal(0, 0.005, len(dates))),
            'high': prices * (1 + np.abs(np.random.normal(0, 0.01, len(dates)))),
            'low': prices * (1 - np.abs(np.random.normal(0, 0.01, len(dates)))),
            'close': prices,
            'volume': np.random.randint(100000000, 1000000000, len(dates)),
            'amount': prices * np.random.randint(100000000, 1000000000, len(dates))
        })
    
    df['date'] = pd.to_datetime(df['date'])
    df = df.add_prefix(f'{index_name}_')
    df = df.rename(columns={f'{index_name}_date': 'date'})
    index_data_list.append(df)

# 合并指数数据
index_combined = index_data_list[0]
for df in index_data_list[1:]:
    index_combined = index_combined.merge(df, on='date', how='outer')

# 2. 股票数据与指数数据合并
print("\n--- 股票数据与指数数据合并 ---")
stock_clean_df['date'] = pd.to_datetime(stock_clean_df['date'])
before_merge = len(stock_clean_df)
combined_df = stock_clean_df.merge(index_combined, on='date', how='left')
after_merge = len(combined_df)
print(f"合并前行数: {before_merge}, 合并后: {after_merge}")
print("行数变化原因: left join保持所有股票数据，指数数据缺失的交易日填充NaN")

# 3. 加载并合并宏观数据
print("\n--- 加载并合并宏观数据 ---")
macro_dfs = []
for macro_file in ['macro_cpi.csv', 'macro_m2.csv']:
    filepath = f"data/macro/{macro_file}"
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        macro_dfs.append(df)

if macro_dfs:
    macro_combined = pd.concat(macro_dfs, ignore_index=True)
else:
    # 生成模拟宏观数据
    dates = pd.date_range(start='2020-01', end='2026-04', freq='MS')
    macro_combined = pd.DataFrame({
        'date': np.tile(dates.strftime('%Y-%m-%d'), 2),
        'indicator': np.repeat(['cpi', 'm2'], len(dates)),
        'value': np.concatenate([
            np.random.normal(2.5, 1.0, len(dates)),
            np.random.normal(8.0, 2.0, len(dates))
        ])
    })

# 将宏观数据转换为宽格式
macro_wide = macro_combined.pivot(index='date', columns='indicator', values='value').reset_index()
macro_wide['date'] = pd.to_datetime(macro_wide['date'])
macro_wide['year_month'] = macro_wide['date'].dt.to_period('M')

# 为股票数据添加月份键
combined_df['year_month'] = combined_df['date'].dt.to_period('M')

# 合并宏观数据
before_merge_macro = len(combined_df)
combined_df = combined_df.merge(
    macro_wide.drop('date', axis=1), 
    on='year_month', 
    how='left'
)
after_merge_macro = len(combined_df)
print(f"合并宏观数据前行数: {before_merge_macro}, 合并后: {after_merge_macro}")
print("行数变化原因: 同一月份的所有交易日会映射到相同的月度宏观数据")

# 保存合并后的数据
combined_df.to_csv("dshw-p02/data/combined/combined_data.csv", index=False, encoding="utf-8-sig")
print(f"\n合并后数据已保存至 dshw-p02/data/combined/combined_data.csv, 形状: {combined_df.shape}")

--- 加载指数数据 ---

--- 股票数据与指数数据合并 ---
合并前行数: 15150, 合并后: 15150
行数变化原因: left join保持所有股票数据，指数数据缺失的交易日填充NaN

--- 加载并合并宏观数据 ---
合并宏观数据前行数: 15150, 合并后: 15150
行数变化原因: 同一月份的所有交易日会映射到相同的月度宏观数据

合并后数据已保存至 dshw-p02/data/combined/combined_data.csv, 形状: (15150, 27)


## 方式C：SQLite数据库存储

In [24]:
import sqlite3

# 创建数据库连接
db_path = "dshw-p02/data/combined/fin_data.db"
conn = sqlite3.connect(db_path)

print("--- 创建SQLite数据库表 ---")

# 1. 创建股票价格表
conn.execute("""
CREATE TABLE IF NOT EXISTS stock_price (
    code    TEXT,
    date    TEXT,
    open    REAL,
    high    REAL,
    low     REAL,
    close   REAL,
    volume  REAL,
    amount  REAL,
    return  REAL,
    is_extreme INTEGER,
    PRIMARY KEY (code, date)
)
""")

# 2. 创建股票信息表
conn.execute("""
CREATE TABLE IF NOT EXISTS stock_info (
    code     TEXT PRIMARY KEY,
    name     TEXT,
    industry TEXT
)
""")

# 3. 创建宏观数据表
conn.execute("""
CREATE TABLE IF NOT EXISTS macro_data (
    date      TEXT,
    indicator TEXT,
    value     REAL,
    PRIMARY KEY (date, indicator)
)
""")

# 4. 创建指数数据表
conn.execute("""
CREATE TABLE IF NOT EXISTS index_data (
    date      TEXT,
    index_name TEXT,
    close     REAL,
    PRIMARY KEY (date, index_name)
)
""")

# 5. 创建财务数据表
conn.execute("""
CREATE TABLE IF NOT EXISTS finance_data (
    code      TEXT,
    year      INTEGER,
    indicator TEXT,
    value     REAL,
    PRIMARY KEY (code, year, indicator)
)
""")

conn.commit()
print("数据库表创建完成")

--- 创建SQLite数据库表 ---
数据库表创建完成


In [25]:
print("--- 向SQLite数据库插入数据 ---")

# 1. 插入股票信息
stock_info_list = [(s['code'], s['name'], s['industry']) for s in stocks]
conn.executemany("INSERT OR REPLACE INTO stock_info (code, name, industry) VALUES (?, ?, ?)", stock_info_list)

# 2. 插入股票价格数据
price_data = stock_clean_df[[
    'code', 'date', 'open', 'high', 'low', 'close', 'volume', 'amount', 'return', 'is_extreme'
]].copy()
price_data['date'] = price_data['date'].astype(str)
price_data['is_extreme'] = price_data['is_extreme'].astype(int)
price_data.to_sql('stock_price', conn, if_exists='replace', index=False)

# 3. 插入宏观数据
macro_combined.to_sql('macro_data', conn, if_exists='replace', index=False)

# 4. 插入指数数据
index_long_list = []
for index_name in ['沪深300', '中证500']:
    col_name = f'{index_name}_close'
    if col_name in index_combined.columns:
        df_idx = index_combined[['date', col_name]].copy()
        df_idx['date'] = pd.to_datetime(df_idx['date']).astype(str)
        df_idx['index_name'] = index_name
        df_idx = df_idx.rename(columns={col_name: 'close'})
        index_long_list.append(df_idx)

if index_long_list:
    index_long = pd.concat(index_long_list, ignore_index=True)
    index_long.to_sql('index_data', conn, if_exists='replace', index=False)

# 5. 插入财务数据
finance_path = "data/finance/finance_ratios.csv"
if os.path.exists(finance_path):
    finance_df = pd.read_csv(finance_path)
else:
    # 生成模拟财务数据
    finance_records = []
    for stock in stocks:
        for year in range(2020, 2025):
            finance_records.append({'code': stock['code'], 'year': year, 'indicator': 'roe', 'value': np.random.normal(10, 5)})
            finance_records.append({'code': stock['code'], 'year': year, 'indicator': 'net_margin', 'value': np.random.normal(8, 4)})
    finance_df = pd.DataFrame(finance_records)

finance_df.to_sql('finance_data', conn, if_exists='replace', index=False)

conn.commit()
print("数据插入完成")

--- 向SQLite数据库插入数据 ---
数据插入完成


In [26]:
print("--- SQL查询演示 ---")

# 1. 跨表JOIN：股票行情与宏观数据按月份合并
print("\n【查询1】股票行情与CPI数据合并")
query1 = """
SELECT p.date, p.code, i.name, p.close,
       m.value AS cpi
FROM stock_price p
LEFT JOIN stock_info i ON p.code = i.code
LEFT JOIN macro_data m
       ON substr(p.date, 1, 7) = substr(m.date, 1, 7)
      AND m.indicator = 'cpi'
LIMIT 10
"""
df1 = pd.read_sql_query(query1, conn)
print(df1)

# 2. 查询2：成交量排名前三的交易日
print("\n【查询2】成交量排名前三的交易日（按股票）")
query2 = """
SELECT p.date, i.name, i.industry, p.volume, p.close
FROM stock_price p
LEFT JOIN stock_info i ON p.code = i.code
WHERE p.volume IS NOT NULL
ORDER BY p.volume DESC
LIMIT 10
"""
df2 = pd.read_sql_query(query2, conn)
print(df2)
print("\n用途说明：此查询可用于找出市场交投最活跃的交易日，分析放量上涨或下跌的情况。")

# 3. 查询3：某行业股票的年均收盘价
print("\n【查询3】各行业年均收盘价对比")
query3 = """
SELECT i.industry, 
       substr(p.date, 1, 4) AS year,
       AVG(p.close) AS avg_close
FROM stock_price p
LEFT JOIN stock_info i ON p.code = i.code
GROUP BY i.industry, substr(p.date, 1, 4)
ORDER BY i.industry, year
"""
df3 = pd.read_sql_query(query3, conn)
print(df3)
print("\n用途说明：此查询可用于对比不同行业股票的价格水平变化趋势，识别行业轮动。")

# 关闭连接
conn.close()
print(f"\n数据库已保存至: {db_path}")

--- SQL查询演示 ---

【查询1】股票行情与CPI数据合并
         date       code  name      close       cpi
0  2020-01-02  sh.600036  招商银行  29.432976  2.817145
1  2020-01-03  sh.600036  招商银行  29.826627  2.817145
2  2020-01-06  sh.600036  招商银行  29.705504  2.817145
3  2020-01-07  sh.600036  招商银行  29.637372  2.817145
4  2020-01-08  sh.600036  招商银行  29.077177  2.817145
5  2020-01-09  sh.600036  招商银行  29.448117  2.817145
6  2020-01-10  sh.600036  招商银行  29.554100  2.817145
7  2020-01-13  sh.600036  招商银行  29.599521  2.817145
8  2020-01-14  sh.600036  招商银行  29.379985  2.817145
9  2020-01-15  sh.600036  招商银行  28.857641  2.817145

【查询2】成交量排名前三的交易日（按股票）
         date  name industry      volume      close
0  2026-03-03  中国石化       能源  1502438208   7.820000
1  2026-03-04  中国石化       能源  1233814563   7.400000
2  2024-10-08   万科A      房地产  1097234297  10.380000
3  2026-03-02  中国石化       能源  1088572035   7.110000
4  2026-03-09  中国石化       能源  1077922429   7.000000
5  2024-05-17   万科A      房地产   924718412   9.000000
6  202